In [1]:
#install four packages into the remote Colab environment
#safetensors reads the tensor-file format used for the model’s pretrained weights.
# psutil lets us inspect RAM usage for our Python process and the machine
#accelerate supports model loading and device placement
#transformers loads model’s architecture, pretrained weights, and tokenizer.
%pip -q install transformers accelerate safetensors psutil

In [2]:
import os
import platform
import importlib.metadata as metadata

import psutil
import torch
from transformers import AutoTokenizer, AutoModelForCausalLM

#l checks our tools, sets a CPU setting, and measures memory before loading the model.
# Use the same thread setting throughout our experiments.
THREADS = min(2, os.cpu_count() or 1)
torch.set_num_threads(THREADS)

# Decimal units: 1 MB = 1,000,000 bytes.
process = psutil.Process(os.getpid())
memory = psutil.virtual_memory()

print("Python:", platform.python_version())
for package in ["torch", "transformers", "accelerate", "psutil"]:
    print(f"{package}: {metadata.version(package)}")

print("\nExperiment device: CPU")
print("PyTorch CPU threads:", torch.get_num_threads())
print(f"System total RAM: {memory.total / 1e9:.2f} GB")
print(f"System available RAM: {memory.available / 1e9:.2f} GB")
print(f"Current Python process RAM: {process.memory_info().rss / 1e6:.1f} MB")

print("\nTensor check:", torch.tensor([1.0, 2.0]) * 2)

Python: 3.13.16
torch: 2.11.0+cpu
transformers: 5.20.1
accelerate: 1.15.0
psutil: 5.9.5

Experiment device: CPU
PyTorch CPU threads: 2
System total RAM: 13.61 GB
System available RAM: 11.63 GB
Current Python process RAM: 530.0 MB

Tensor check: tensor([2., 4.])


In [3]:
MODEL_ID = "HuggingFaceTB/SmolLM2-135M"

# Record current process memory before loading.
ram_before = process.memory_info().rss

# Download and load the tokenizer.
tokenizer = AutoTokenizer.from_pretrained(MODEL_ID)

# Download and load the original model in FP32 on the CPU.
model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID,
    dtype=torch.float32,
    device_map="cpu",
)

# Set inference behaviour and disable parameter gradients.
model.eval()
model.requires_grad_(False)

# Count unique parameters and the bytes they occupy.
parameter_count = sum(p.numel() for p in model.parameters())
parameter_bytes = sum(
    p.numel() * p.element_size()
    for p in model.parameters()
)

ram_after = process.memory_info().rss

print(f"Model: {MODEL_ID}")
print(f"Parameter count: {parameter_count:,}")
print(f"Parameter storage: {parameter_bytes / 1e6:.1f} MB")
print(f"Process RAM before loading: {ram_before / 1e6:.1f} MB")
print(f"Process RAM after loading: {ram_after / 1e6:.1f} MB")
print(f"Change in process RAM: {(ram_after - ram_before) / 1e6:.1f} MB")
print(f"Model device: {next(model.parameters()).device}")
print(f"Model dtype: {next(model.parameters()).dtype}")

Loading weights:   0%|          | 0/272 [00:00<?, ?it/s]

Model: HuggingFaceTB/SmolLM2-135M
Parameter count: 134,515,008
Parameter storage: 538.1 MB
Process RAM before loading: 537.6 MB
Process RAM after loading: 1143.8 MB
Change in process RAM: 606.2 MB
Model device: cpu
Model dtype: torch.float32


In [4]:
#inspects the model and lists the layers we’ll quantize. It doesn’t change any weights.
# Find internal linear layers, excluding the final output projection.
output_layer = model.get_output_embeddings()

target_layers = {
    name: layer
    for name, layer in model.named_modules()
    if isinstance(layer, torch.nn.Linear)
    and layer is not output_layer
}

# Count weights that we plan to quantize.
target_weight_count = sum(
    layer.weight.numel()
    for layer in target_layers.values()
)

# Everything else will remain FP32.
remaining_parameter_count = parameter_count - target_weight_count

print(f"Internal linear layers: {len(target_layers)}")
print(f"Weights to quantize: {target_weight_count:,}")
print(f"Parameters remaining FP32: {remaining_parameter_count:,}")
print(
    f"Percentage of parameters targeted: "
    f"{100 * target_weight_count / parameter_count:.1f}%"
)

print("\nFirst seven target layers:")
for name, layer in list(target_layers.items())[:7]:
    print(f"{name}: {tuple(layer.weight.shape)}")

# Estimate parameter storage before metadata and file overhead.
print("\nEstimated parameter storage:")
for label, bits in [("FP32", 32), ("INT8", 8), ("Packed INT4", 4)]:
    estimated_bytes = (
        target_weight_count * bits / 8
        + remaining_parameter_count * 4
    )
    print(f"{label}: {estimated_bytes / 1e6:.1f} MB")

print(
    "\nInput embedding and output projection share weights:",
    model.get_input_embeddings().weight is output_layer.weight,
)

Internal linear layers: 210
Weights to quantize: 106,168,320
Parameters remaining FP32: 28,346,688
Percentage of parameters targeted: 78.9%

First seven target layers:
model.layers.0.self_attn.q_proj: (576, 576)
model.layers.0.self_attn.k_proj: (192, 576)
model.layers.0.self_attn.v_proj: (192, 576)
model.layers.0.self_attn.o_proj: (576, 576)
model.layers.0.mlp.gate_proj: (1536, 576)
model.layers.0.mlp.up_proj: (1536, 576)
model.layers.0.mlp.down_proj: (576, 1536)

Estimated parameter storage:
FP32: 538.1 MB
INT8: 219.6 MB
Packed INT4: 166.5 MB

Input embedding and output projection share weights: True


In [5]:
%%writefile /content/quantizer.py
import torch
import torch.nn.functional as F


def quantize_symmetric(weight, bits=8, granularity="channel"):
    if bits not in (4, 8):
        raise ValueError("bits must be 4 or 8")
    if granularity not in ("tensor", "channel"):
        raise ValueError("Unsupported granularity")
    if weight.ndim != 2:
        raise ValueError("Expected a 2D weight matrix")

    weight = weight.detach().float()
    qmax = 2 ** (bits - 1) - 1

    if granularity == "channel":
        max_abs = weight.abs().amax(dim=1, keepdim=True)
    else:
        max_abs = weight.abs().amax()

    scale = torch.where(
        max_abs == 0,
        torch.ones_like(max_abs),
        max_abs / qmax,
    )
    codes = torch.round(weight / scale).clamp(-qmax, qmax)
    return codes.to(torch.int8), scale


def quantize_asymmetric(weight, bits=4, granularity="channel"):
    if bits not in (4, 8):
        raise ValueError("bits must be 4 or 8")
    if granularity not in ("tensor", "channel"):
        raise ValueError("Unsupported granularity")
    if weight.ndim != 2:
        raise ValueError("Expected a 2D weight matrix")

    weight = weight.detach().float()
    qmax = 2 ** bits - 1

    if granularity == "channel":
        minimum = weight.amin(dim=1, keepdim=True)
        maximum = weight.amax(dim=1, keepdim=True)
    else:
        minimum = weight.amin()
        maximum = weight.amax()

    minimum = torch.minimum(minimum, torch.zeros_like(minimum))
    maximum = torch.maximum(maximum, torch.zeros_like(maximum))
    span = maximum - minimum

    scale = torch.where(
        span == 0, torch.ones_like(span), span / qmax
    )
    zero_point = torch.round(-minimum / scale).clamp(0, qmax)
    codes = (torch.round(weight / scale) + zero_point).clamp(0, qmax)
    return codes.to(torch.uint8), scale, zero_point.to(torch.uint8)


def pack_int4(codes, symmetric=True):
    flat = codes.detach().reshape(-1).to(torch.int16)
    lower, upper = (-7, 7) if symmetric else (0, 15)

    if not ((flat >= lower) & (flat <= upper)).all().item():
        raise ValueError("Codes outside the allowed INT4 range")

    if symmetric:
        flat = flat + 7
    flat = flat.to(torch.uint8)

    if flat.numel() % 2:
        flat = torch.cat([flat, flat.new_zeros(1)])

    return flat[0::2] | (flat[1::2] << 4)


def unpack_int4(packed, shape, symmetric=True):
    count = 1
    for dimension in shape:
        count *= dimension

    if packed.dtype != torch.uint8 or packed.ndim != 1:
        raise ValueError("Expected a one-dimensional uint8 tensor")
    if packed.numel() != (count + 1) // 2:
        raise ValueError("Packed length does not match shape")

    flat = torch.empty(
        packed.numel() * 2,
        dtype=torch.uint8,
        device=packed.device,
    )
    flat[0::2] = packed & 15
    flat[1::2] = (packed >> 4) & 15
    flat = flat[:count]

    if symmetric:
        flat = (flat.to(torch.int16) - 7).to(torch.int8)
    return flat.reshape(shape)


class QuantizedLinear(torch.nn.Module):
    def __init__(
        self, weight, bias=None, bits=4,
        granularity="channel", symmetric=True, collect_stats=False,
    ):
        super().__init__()
        self.bits = bits
        self.granularity = granularity
        self.symmetric = symmetric
        self.out_features, self.in_features = weight.shape
        self.weight_shape = tuple(weight.shape)

        with torch.no_grad():
            if symmetric:
                q, scale = quantize_symmetric(weight, bits, granularity)
                zero_point = None
            else:
                q, scale, zero_point = quantize_asymmetric(
                    weight, bits, granularity
                )

            self.conversion_stats = None
            if collect_stats:
                original = weight.detach().float()
                reconstructed = q.float()
                if not symmetric:
                    reconstructed.sub_(zero_point.float())
                reconstructed.mul_(scale)
                difference = reconstructed - original

                self.conversion_stats = {
                    "squared_error": difference.square().sum(
                        dtype=torch.float64
                    ).item(),
                    "squared_weight": original.square().sum(
                        dtype=torch.float64
                    ).item(),
                    "weight_count": original.numel(),
                }
                del reconstructed, difference, original

            stored = pack_int4(q, symmetric) if bits == 4 else q
            self.register_buffer("weight_codes", stored)
            self.register_buffer("scale", scale)
            self.register_buffer("zero_point", zero_point)
            self.register_buffer(
                "bias",
                None if bias is None else bias.detach().float().clone(),
            )

    def reconstruct_weight(self):
        q = (
            unpack_int4(
                self.weight_codes, self.weight_shape, self.symmetric
            )
            if self.bits == 4 else self.weight_codes
        )
        weight = q.float()
        if not self.symmetric:
            weight.sub_(self.zero_point.float())
        return weight.mul_(self.scale)

    def forward(self, inputs):
        return F.linear(inputs, self.reconstruct_weight(), self.bias)


CONFIGS = {
    "int8_channel_sym": dict(
        bits=8, granularity="channel", symmetric=True
    ),
    "int4_channel_sym": dict(
        bits=4, granularity="channel", symmetric=True
    ),
    "int4_tensor_sym": dict(
        bits=4, granularity="tensor", symmetric=True
    ),
    "int4_channel_asym": dict(
        bits=4, granularity="channel", symmetric=False
    ),
}


def convert_linear_layers(model_to_convert, settings):
    output_layer = model_to_convert.get_output_embeddings()
    names = [
        name for name, layer in model_to_convert.named_modules()
        if isinstance(layer, torch.nn.Linear)
        and layer is not output_layer
    ]

    total_error = 0.0
    total_weight = 0.0
    total_count = 0
    layer_results = []

    with torch.inference_mode():
        for name in names:
            original = model_to_convert.get_submodule(name)
            replacement = QuantizedLinear(
                original.weight, original.bias,
                collect_stats=True, **settings,
            )
            stats = replacement.conversion_stats
            count = stats["weight_count"]
            total_error += stats["squared_error"]
            total_weight += stats["squared_weight"]
            total_count += count
            layer_results.append({
                "layer": name,
                "weight_count": count,
                "mse": stats["squared_error"] / count,
            })
            del replacement.conversion_stats

            parent_name, _, child_name = name.rpartition(".")
            parent = (
                model_to_convert.get_submodule(parent_name)
                if parent_name else model_to_convert
            )
            setattr(parent, child_name, replacement)
            del original, replacement

    return {
        "layers_quantized": len(names),
        "weights_quantized": total_count,
        "weight_mse": total_error / total_count if total_count else 0.0,
        "relative_squared_error": (
            total_error / total_weight if total_weight else 0.0
        ),
        "per_layer": layer_results,
    }

Overwriting /content/quantizer.py


In [6]:
import sys
import importlib
import json
import hashlib
from pathlib import Path

PROJECT_DIR = Path("/content/model_quantizer")
BASELINE_DIR = PROJECT_DIR / "baseline_fp32"
module_path = Path("/content/quantizer.py")
PROJECT_DIR.mkdir(parents=True, exist_ok=True)

# Use the directly written module as our source of truth.
if "/content" not in sys.path:
    sys.path.insert(0, "/content")

# Avoid reusing an older module imported from another location.
sys.modules.pop("quantizer", None)
importlib.invalidate_caches()
import quantizer

assert Path(quantizer.__file__).resolve() == module_path.resolve()

quantize_symmetric = quantizer.quantize_symmetric
quantize_asymmetric = quantizer.quantize_asymmetric
pack_int4 = quantizer.pack_int4
unpack_int4 = quantizer.unpack_int4
QuantizedLinear = quantizer.QuantizedLinear
convert_linear_layers = quantizer.convert_linear_layers
CONFIGS = quantizer.CONFIGS

# Confirm that the notebook model is still the FP32 baseline.
assert not any(
    isinstance(layer, QuantizedLinear)
    for layer in model.modules()
)
assert all(
    parameter.dtype == torch.float32
    for parameter in model.parameters()
)

# Reuse the baseline already saved in this session.
if not (
    (BASELINE_DIR / "config.json").exists()
    and list(BASELINE_DIR.glob("*.safetensors"))
):
    model.save_pretrained(BASELINE_DIR, safe_serialization=True)
    tokenizer.save_pretrained(BASELINE_DIR)
    print("FP32 baseline saved.")
else:
    print("Existing FP32 baseline reused.")

versions = {
    package: metadata.version(package)
    for package in [
        "torch", "transformers", "accelerate",
        "safetensors", "psutil"
    ]
}

(PROJECT_DIR / "environment.json").write_text(
    json.dumps(
        {
            "versions": versions,
            "cpu_threads": THREADS,
            "quantizer_sha256": hashlib.sha256(
                module_path.read_bytes()
            ).hexdigest(),
        },
        indent=2,
    ),
    encoding="utf-8",
)

checkpoint_bytes = sum(
    path.stat().st_size
    for path in BASELINE_DIR.glob("*.safetensors")
)

print("Imported module:", quantizer.__file__)
print("Configurations:", list(CONFIGS))
print(f"FP32 checkpoint: {checkpoint_bytes / 1e6:.1f} MB")
print("Environment saved. No notebook-history extraction used.")

Existing FP32 baseline reused.
Imported module: /content/quantizer.py
Configurations: ['int8_channel_sym', 'int4_channel_sym', 'int4_tensor_sym', 'int4_channel_asym']
FP32 checkpoint: 538.1 MB
Environment saved. No notebook-history extraction used.


In [7]:
%pip -q install datasets

import json
import math
from datasets import load_dataset

SEQ_LEN = 128
NUM_BLOCKS = 16
REQUIRED_TOKENS = SEQ_LEN * NUM_BLOCKS

# Stream a small amount of held-out text instead of loading the full dataset.
dataset = load_dataset(
    "Salesforce/wikitext",
    "wikitext-2-raw-v1",
    split="test",
    streaming=True,
)

token_ids = []
rows_read = 0

for row in dataset:
    rows_read += 1
    text = row["text"].strip()
    if not text:
        continue

    token_ids.extend(
        tokenizer.encode(text + "\n\n", add_special_tokens=False)
    )

    if len(token_ids) >= REQUIRED_TOKENS:
        break

if len(token_ids) < REQUIRED_TOKENS:
    raise RuntimeError("Not enough evaluation tokens")

eval_blocks = torch.tensor(
    token_ids[:REQUIRED_TOKENS],
    dtype=torch.long,
).reshape(NUM_BLOCKS, SEQ_LEN)

timing_prompt = "Once upon a time, a young scientist discovered"
timing_inputs = tokenizer(
    timing_prompt,
    return_tensors="pt",
    add_special_tokens=False,
)

torch.save(
    {
        "eval_blocks": eval_blocks,
        "timing_input_ids": timing_inputs["input_ids"],
        "timing_attention_mask": timing_inputs["attention_mask"],
    },
    PROJECT_DIR / "workload.pt",
)

workload_settings = {
    "dataset": "Salesforce/wikitext",
    "dataset_config": "wikitext-2-raw-v1",
    "split": "test",
    "selection": "First nonempty rows; tokenize each with two trailing newlines",
    "rows_read": rows_read,
    "sequence_length": SEQ_LEN,
    "blocks": NUM_BLOCKS,
    "input_tokens": REQUIRED_TOKENS,
    "predicted_tokens": NUM_BLOCKS * (SEQ_LEN - 1),
    "batch_size": 1,
    "timing_prompt": timing_prompt,
    "generation_tokens": 50,
    "cpu_threads": THREADS,
}
(PROJECT_DIR / "workload.json").write_text(
    json.dumps(workload_settings, indent=2),
    encoding="utf-8",
)

# A one-block baseline pilot, not the final full evaluation.
model.eval()
with torch.inference_mode():
    pilot_output = model(
        input_ids=eval_blocks[:1],
        labels=eval_blocks[:1],
        use_cache=False,
    )
    pilot_loss = pilot_output.loss.item()

assert math.isfinite(pilot_loss)
del pilot_output

print("Baseline inference passed.")
print(f"Pilot loss: {pilot_loss:.4f}")
print(f"Pilot perplexity: {math.exp(pilot_loss):.2f}")
print("Evaluation blocks:", tuple(eval_blocks.shape))
print("Total predicted tokens:", NUM_BLOCKS * (SEQ_LEN - 1))
print("Workload saved. Do not change it between configurations.")
print("Final RAM measurements will use fresh processes.")

Baseline inference passed.
Pilot loss: 3.0615
Pilot perplexity: 21.36
Evaluation blocks: (16, 128)
Total predicted tokens: 2032
Workload saved. Do not change it between configurations.
Final RAM measurements will use fresh processes.


In [8]:
# Verify the actual module we will use for conversion and deployment.
with torch.inference_mode():
    # Exact packing/unpacking, including odd-length input.
    for symmetric, original in [
        (True, torch.arange(-7, 8, dtype=torch.int8)),
        (False, torch.arange(16, dtype=torch.uint8)),
        (False, torch.tensor([0, 7, 15], dtype=torch.uint8)),
    ]:
        packed = quantizer.pack_int4(original, symmetric)
        restored = quantizer.unpack_int4(
            packed, original.shape, symmetric
        )
        assert torch.equal(original, restored)

    test_weights = torch.tensor([
        [0.0, 0.0, 0.0],
        [-0.7, 0.23, 0.46],
        [-1.4, 0.43, 0.92],
    ])
    test_bias = torch.tensor([0.1, -0.2, 0.3])
    test_inputs = torch.tensor([[1.0, 2.0, -1.0]])

    for name, settings in CONFIGS.items():
        test_layer = QuantizedLinear(
            test_weights,
            test_bias,
            collect_stats=True,
            **settings,
        )
        restored_weights = test_layer.reconstruct_weight()

        assert torch.isfinite(restored_weights).all()
        assert torch.equal(restored_weights[0], torch.zeros(3))

        if settings["symmetric"]:
            assert test_layer.zero_point is None
        else:
            assert test_layer.zero_point is not None

        if settings["granularity"] == "channel":
            assert test_layer.scale.shape == (3, 1)
        else:
            assert test_layer.scale.numel() == 1

        # Check error statistics against reconstruction after packing.
        measured_error = (
            restored_weights - test_weights
        ).square().sum(dtype=torch.float64).item()

        assert abs(
            measured_error
            - test_layer.conversion_stats["squared_error"]
        ) < 1e-10

        torch.testing.assert_close(
            test_layer(test_inputs),
            torch.nn.functional.linear(
                test_inputs, restored_weights, test_bias
            ),
        )

        assert not list(test_layer.parameters())
        print(f"{name}: passed")


class ConversionTestModel(torch.nn.Module):
    def __init__(self):
        super().__init__()
        self.hidden = torch.nn.Linear(3, 3)
        self.output = torch.nn.Linear(3, 2)

    def get_output_embeddings(self):
        return self.output


for name, settings in CONFIGS.items():
    test_model = ConversionTestModel()
    preserved_output = test_model.output
    stats = convert_linear_layers(test_model, settings)

    assert isinstance(test_model.hidden, QuantizedLinear)
    assert test_model.output is preserved_output
    assert stats["layers_quantized"] == 1
    assert stats["weights_quantized"] == 9
    assert not hasattr(test_model.hidden, "conversion_stats")

del test_layer, restored_weights, test_model, preserved_output
print("\nAll checks passed. Ready for full-model conversion.")

int8_channel_sym: passed
int4_channel_sym: passed
int4_tensor_sym: passed
int4_channel_asym: passed

All checks passed. Ready for full-model conversion.


In [16]:
%%writefile /content/checkpoints.py
import json
from pathlib import Path

import torch
from safetensors.torch import save_file, load_file
from transformers import AutoConfig, AutoModelForCausalLM

from quantizer import QuantizedLinear


def save_quantized(model, folder, settings, statistics):
    folder = Path(folder)
    folder.mkdir(parents=True, exist_ok=True)
    model.config.save_pretrained(folder)

    state = model.state_dict()
    if not model.config.tie_word_embeddings:
        raise ValueError("Expected tied embeddings")

    embedding_key = "model.embed_tokens.weight"
    output_key = "lm_head.weight"

    if state[embedding_key].data_ptr() != state[output_key].data_ptr():
        raise ValueError("Embedding/output weights are not shared")

    stored_state = {
        name: tensor.detach().cpu().contiguous()
        for name, tensor in state.items()
        if name != output_key
    }
    save_file(stored_state, str(folder / "model.safetensors"))

    manifest = {
        "format_version": 1,
        "settings": settings,
        "statistics": statistics,
        "layers": {
            name: {"weight_shape": list(layer.weight_shape)}
            for name, layer in model.named_modules()
            if isinstance(layer, QuantizedLinear)
        },
    }
    (folder / "quantization.json").write_text(
        json.dumps(manifest, indent=2),
        encoding="utf-8",
    )


def load_quantized(folder):
    folder = Path(folder)
    manifest = json.loads(
        (folder / "quantization.json").read_text(encoding="utf-8")
    )
    if manifest["format_version"] != 1:
        raise ValueError("Unsupported checkpoint version")

    state = load_file(str(folder / "model.safetensors"), device="cpu")
    config = AutoConfig.from_pretrained(folder, local_files_only=True)

    with torch.device("meta"):
        model = AutoModelForCausalLM.from_config(
            config, dtype=torch.float32
        )

    # Rebuild non-checkpoint rotary-position buffers on CPU.
    for name, module in list(model.named_modules()):
        if name.endswith("rotary_emb"):
            with torch.device("cpu"):
              replacement = type(module)(config=config)
            parent_name, _, child_name = name.rpartition(".")
            parent = (
                model.get_submodule(parent_name)
                if parent_name else model
            )
            setattr(parent, child_name, replacement)

    settings = manifest["settings"]

    for name, information in manifest["layers"].items():
        layer = QuantizedLinear.__new__(QuantizedLinear)
        torch.nn.Module.__init__(layer)

        layer.bits = settings["bits"]
        layer.granularity = settings["granularity"]
        layer.symmetric = settings["symmetric"]
        layer.weight_shape = tuple(information["weight_shape"])
        layer.out_features, layer.in_features = layer.weight_shape

        for buffer_name in ["weight_codes", "scale", "zero_point", "bias"]:
            key = f"{name}.{buffer_name}"
            buffer = (
                torch.empty_like(state[key], device="meta")
                if key in state else None
            )
            layer.register_buffer(buffer_name, buffer)

        parent_name, _, child_name = name.rpartition(".")
        parent = (
            model.get_submodule(parent_name)
            if parent_name else model
        )
        setattr(parent, child_name, layer)

    state["lm_head.weight"] = state["model.embed_tokens.weight"]
    model.load_state_dict(state, strict=True, assign=True)
    model.tie_weights()
    model.eval()
    model.requires_grad_(False)

    meta_names = [
        name
        for name, tensor in list(model.named_parameters())
        + list(model.named_buffers())
        if tensor.is_meta
    ]
    if meta_names:
        raise RuntimeError(f"Unloaded meta tensors: {meta_names}")

    if model.get_input_embeddings().weight is not (
        model.get_output_embeddings().weight
    ):
        raise RuntimeError("Weight sharing was not restored")

    return model

Overwriting /content/checkpoints.py


In [17]:
import subprocess
import sys

verification_script = r'''
import json
from pathlib import Path

import torch

from checkpoints import load_quantized

root = Path("/content/model_quantizer")
environment = json.loads((root / "environment.json").read_text())
torch.set_num_threads(environment["cpu_threads"])

print("Loading existing INT4 checkpoint...", flush=True)
restored = load_quantized(root / "int4_channel_sym")

workload = torch.load(
    root / "workload.pt",
    map_location="cpu",
    weights_only=True,
)
probe = workload["eval_blocks"][:1, :16]

with torch.inference_mode():
    output = restored(input_ids=probe, use_cache=False)
    logits = output.logits[:, -1, :]
    assert torch.isfinite(logits).all()

print("PASS: checkpoint loaded and inference produced finite outputs.")
print("Logits shape:", tuple(logits.shape))
print("Device:", next(restored.parameters()).device)
print("No meta parameters or buffers remain.")
'''

result = subprocess.run(
    [sys.executable, "-u", "-c", verification_script],
    cwd="/content",
    capture_output=True,
    text=True,
)

print(result.stdout)
if result.returncode:
    print("ERROR OUTPUT:\n", result.stderr)
else:
    print("Exit code: 0")

Loading existing INT4 checkpoint...
PASS: checkpoint loaded and inference produced finite outputs.
Logits shape: (1, 49152)
Device: cpu
No meta parameters or buffers remain.

Exit code: 0


In [18]:
import subprocess
import sys
from pathlib import Path

builder_path = Path("/content/build_quantized.py")
assert builder_path.exists(), "The conversion script is missing."

for name in CONFIGS:
    print(f"\n{'=' * 55}\nBuilding: {name}", flush=True)

    result = subprocess.run(
        [sys.executable, "-u", str(builder_path), name],
        cwd="/content",
        capture_output=True,
        text=True,
    )

    print(result.stdout, flush=True)

    if result.returncode != 0:
        print("ERROR OUTPUT:\n", result.stderr)
        raise RuntimeError(f"Build failed for {name}")

print("All four configurations saved and reload comparisons passed.")


Building: int8_channel_sym
Loading FP32 baseline for int8_channel_sym...
Quantizing internal linear layers...
Saving quantized checkpoint...
Reloading compressed checkpoint...

Configuration: int8_channel_sym
Layers quantized: 210
Weights quantized: 106,168,320
Weight MSE: 2.6983758e-06
Checkpoint size: 220.23 MB
Reload maximum logit difference: 0
PASS: quantization, inference, saving, and reloading.
Peak inference RAM and perplexity are not measured yet.


Building: int4_channel_sym
Loading FP32 baseline for int4_channel_sym...
Quantizing internal linear layers...
Saving quantized checkpoint...
Reloading compressed checkpoint...

Configuration: int4_channel_sym
Layers quantized: 210
Weights quantized: 106,168,320
Weight MSE: 0.00088296281
Checkpoint size: 167.15 MB
Reload maximum logit difference: 0
PASS: quantization, inference, saving, and reloading.
Peak inference RAM and perplexity are not measured yet.


Building: int4_tensor_sym
Loading FP32 baseline for int4_tensor_sym...
Quan

In [19]:
%%writefile /content/benchmark.py
import argparse
import gc
import json
import math
import platform
import statistics
import threading
import time
from pathlib import Path

import psutil
import torch
from transformers import AutoModelForCausalLM

from checkpoints import load_quantized

ROOT = Path("/content/model_quantizer")
PROCESS = psutil.Process()


class MemorySampler:
    """Sample this process's physical RAM every 5 milliseconds."""
    def __init__(self):
        self.peak = PROCESS.memory_info().rss
        self.stop_event = threading.Event()
        self.thread = threading.Thread(target=self.sample, daemon=True)

    def sample(self):
        while not self.stop_event.is_set():
            self.peak = max(self.peak, PROCESS.memory_info().rss)
            self.stop_event.wait(0.005)

    def __enter__(self):
        self.thread.start()
        return self

    def __exit__(self, *args):
        self.peak = max(self.peak, PROCESS.memory_info().rss)
        self.stop_event.set()
        self.thread.join()


def main():
    parser = argparse.ArgumentParser()
    parser.add_argument("name")
    parser.add_argument("mode", choices=["quality_ram", "speed"])
    args = parser.parse_args()

    environment = json.loads((ROOT / "environment.json").read_text())
    torch.set_num_threads(environment["cpu_threads"])
    torch.manual_seed(42)

    workload = torch.load(
        ROOT / "workload.pt",
        map_location="cpu",
        weights_only=True,
    )
    framework_ram = PROCESS.memory_info().rss
    folder = ROOT / args.name

    with MemorySampler() as loading_memory:
        if args.name == "baseline_fp32":
            model = AutoModelForCausalLM.from_pretrained(
                folder,
                dtype=torch.float32,
                device_map="cpu",
                local_files_only=True,
            )
        else:
            model = load_quantized(folder)

        model.eval()
        model.requires_grad_(False)

    gc.collect()
    loaded_ram = PROCESS.memory_info().rss

    input_ids = workload["timing_input_ids"]
    attention_mask = workload["timing_attention_mask"]
    generation_settings = {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "max_new_tokens": 50,
        "min_new_tokens": 50,
        "do_sample": False,
        "num_beams": 1,
        "use_cache": True,
        "pad_token_id": model.config.eos_token_id,
    }

    result = {
        "configuration": args.name,
        "mode": args.mode,
        "cpu_threads": torch.get_num_threads(),
        "platform": platform.platform(),
        "cpu_description": platform.processor(),
        "total_system_ram_mb": psutil.virtual_memory().total / 1e6,
        "checkpoint_mb": sum(
            path.stat().st_size
            for path in folder.glob("*.safetensors")
        ) / 1e6,
    }

    with torch.inference_mode():
        if args.mode == "quality_ram":
            total_nll = 0.0
            predicted_tokens = 0

            # Include first-use allocations: no inference warmup here.
            with MemorySampler() as inference_memory:
                for block in workload["eval_blocks"]:
                    tokens = block.unsqueeze(0)
                    output = model(
                        input_ids=tokens,
                        labels=tokens,
                        use_cache=False,
                    )
                    loss = output.loss.item()
                    if not math.isfinite(loss):
                        raise RuntimeError("Non-finite evaluation loss")

                    count = tokens.shape[1] - 1
                    total_nll += loss * count
                    predicted_tokens += count
                    del output

                generated = model.generate(**generation_settings)
                assert generated.shape[1] - input_ids.shape[1] == 50
                del generated

            mean_loss = total_nll / predicted_tokens
            result.update({
                "framework_ram_mb": framework_ram / 1e6,
                "loaded_ram_mb": loaded_ram / 1e6,
                "peak_loading_ram_mb": loading_memory.peak / 1e6,
                "peak_inference_ram_mb": inference_memory.peak / 1e6,
                "memory_sampling_interval_ms": 5,
                "meets_1gb_budget": inference_memory.peak < 1_000_000_000,
                "evaluated_tokens": predicted_tokens,
                "mean_nll": mean_loss,
                "perplexity": math.exp(mean_loss),
            })

        else:
            # Warmup is excluded from timing.
            warmup = model.generate(**generation_settings)
            del warmup

            durations = []
            for _ in range(3):
                start = time.perf_counter()
                generated = model.generate(**generation_settings)
                duration = time.perf_counter() - start
                assert generated.shape[1] - input_ids.shape[1] == 50
                durations.append(duration)
                del generated

            median_seconds = statistics.median(durations)
            result.update({
                "generation_tokens": 50,
                "timing_trials_seconds": durations,
                "median_generation_seconds": median_seconds,
                "tokens_per_second": 50 / median_seconds,
            })

    results_dir = ROOT / "results"
    results_dir.mkdir(exist_ok=True)
    destination = results_dir / f"{args.name}_{args.mode}.json"
    destination.write_text(json.dumps(result, indent=2), encoding="utf-8")
    print(json.dumps(result, indent=2), flush=True)


if __name__ == "__main__":
    main()

Writing /content/benchmark.py


In [20]:
import json
import subprocess
import sys

configurations = ["baseline_fp32", *CONFIGS.keys()]

for name in configurations:
    for mode in ["quality_ram", "speed"]:
        print(f"\nRunning {name}: {mode} ...", flush=True)

        completed = subprocess.run(
            [
                sys.executable,
                "-u",
                "/content/benchmark.py",
                name,
                mode,
            ],
            cwd="/content",
            capture_output=True,
            text=True,
        )

        if completed.returncode != 0:
            print(completed.stdout)
            print(completed.stderr)
            raise RuntimeError(f"Benchmark failed: {name}, {mode}")

        result_path = (
            PROJECT_DIR / "results" / f"{name}_{mode}.json"
        )
        measurement = json.loads(
            result_path.read_text(encoding="utf-8")
        )

        if mode == "quality_ram":
            print(
                f"Perplexity: {measurement['perplexity']:.3f} | "
                f"Peak inference RAM: "
                f"{measurement['peak_inference_ram_mb']:.1f} MB | "
                f"Below 1 GB: {measurement['meets_1gb_budget']}",
                flush=True,
            )
        else:
            print(
                f"50-token generation: "
                f"{measurement['median_generation_seconds']:.3f} s | "
                f"{measurement['tokens_per_second']:.2f} tokens/s",
                flush=True,
            )

print("\nAll benchmarks finished. Results saved in:", PROJECT_DIR / "results")


Running baseline_fp32: quality_ram ...
Perplexity: 36.203 | Peak inference RAM: 1105.4 MB | Below 1 GB: False

Running baseline_fp32: speed ...
50-token generation: 3.700 s | 13.51 tokens/s

Running int8_channel_sym: quality_ram ...
Perplexity: 36.585 | Peak inference RAM: 788.7 MB | Below 1 GB: True

Running int8_channel_sym: speed ...
50-token generation: 8.704 s | 5.74 tokens/s

Running int4_channel_sym: quality_ram ...
Perplexity: 81.945 | Peak inference RAM: 730.7 MB | Below 1 GB: True

Running int4_channel_sym: speed ...
50-token generation: 23.265 s | 2.15 tokens/s

Running int4_tensor_sym: quality_ram ...
Perplexity: 4976543.760 | Peak inference RAM: 729.6 MB | Below 1 GB: True

Running int4_tensor_sym: speed ...
50-token generation: 20.564 s | 2.43 tokens/s

Running int4_channel_asym: quality_ram ...
Perplexity: 65.510 | Peak inference RAM: 730.6 MB | Below 1 GB: True

Running int4_channel_asym: speed ...
50-token generation: 20.419 s | 2.45 tokens/s

All benchmarks finished.

In [21]:
from pathlib import Path
import zipfile
from google.colab import files

root = Path("/content/model_quantizer")
backup_path = Path("/content/SmolLM2_Quantizer_Backup.zip")

scripts = [
    Path("/content/quantizer.py"),
    Path("/content/checkpoints.py"),
    Path("/content/build_quantized.py"),
    Path("/content/benchmark.py"),
]

for path in scripts:
    assert path.exists(), f"Missing script: {path.name}"

for name in CONFIGS:
    assert (root / name / "model.safetensors").exists(), (
        f"Missing checkpoint: {name}"
    )

assert len(list((root / "results").glob("*.json"))) >= 10, (
    "Expected quality/RAM and speed results for all five models"
)

with zipfile.ZipFile(
    backup_path, "w", compression=zipfile.ZIP_STORED
) as archive:
    for path in scripts:
        archive.write(path, f"SmolLM2_Quantizer/{path.name}")

    for path in sorted(root.rglob("*")):
        if not path.is_file():
            continue

        relative = path.relative_to(root)

        # Omit only the downloadable FP32 weight files.
        if (
            relative.parts[0] == "baseline_fp32"
            and path.suffix in {".safetensors", ".bin"}
        ):
            continue

        # Ignore temporary Python cache files.
        if "__pycache__" in relative.parts:
            continue

        archive.write(
            path,
            f"SmolLM2_Quantizer/artifacts/{relative.as_posix()}",
        )

print(f"Backup ready: {backup_path.stat().st_size / 1e6:.1f} MB")
print("Contains code, evaluation inputs, results, and four quantized models.")
print("Also save your notebook using Colab: File → Download → .ipynb")
files.download(str(backup_path))

Backup ready: 725.0 MB
Contains code, evaluation inputs, results, and four quantized models.
Also save your notebook using Colab: File → Download → .ipynb


<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>